# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"



13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
import os
from typing import List
from pydantic import BaseModel, Field
from openai import OpenAI

# 1. The Schema (Required for Pydantic BaseModel output)
class ArticleAnalysis(BaseModel):
    author: str
    title: str
    relevance: str = Field(description="Relevance for an AI professional (one paragraph).")
    summary: str = Field(description="Succinct summary no longer than 1000 tokens.")
    tone: str
    input_tokens: int = 0
    output_tokens: int = 0

# 2. The Processing Function
def analyze_article_to_pydantic(text_content: str, chosen_style: str):
    # Initialize client with your specific Gateway settings
    client = OpenAI(
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
    )

    # Separate Instructions and Context
    instruction_template = (
        "You are a professional analyst. Produce a structured output. "
        "The summary must be written in the following tone: {tone}. "
        "Ensure the relevance field explains why this is important for an AI professional."
    )
    context_template = "Article Content:\n\n{content}"

    # Dynamic Formatting
    dev_message = instruction_template.format(tone=chosen_style)
    user_message = context_template.format(content=text_content)

    # API Call (using a non-GPT-5 model like gpt-4o-mini)
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "developer", "content": dev_message},
            {"role": "user", "content": user_message},
        ],
        response_format=ArticleAnalysis,
    )

    # Extract the parsed Pydantic object
    analysis = completion.choices[0].message.parsed
    
    # Map the usage metadata from the response object into the Pydantic fields
    analysis.input_tokens = completion.usage.prompt_tokens
    analysis.output_tokens = completion.usage.completion_tokens

    return analysis

# --- Usage Example ---
# Assuming 'full_text' is the content extracted from your PDF
# Tone choice: "Bureaucratese"
result_object = analyze_article_to_pydantic(document_text, "Bureaucratese")

print(f"Analysis for: {result_object.title}")
print(f"Summary in {result_object.tone}:\n{result_object.summary}")
print(f"Tokens Used - In: {result_object.input_tokens}, Out: {result_object.output_tokens}")

Analysis for: Managing Oneself
Summary in Bureaucratese:
In "Managing Oneself," Peter Drucker posits that success in today’s knowledge economy is predicated on the individual’s understanding of themselves as professionals, necessitating a shift in approach wherein individuals take on the role of their own chief executive officers. In an era where career management falls predominantly on the individual rather than the organization, knowing one's strengths, weaknesses, preferred work style, values, and optimal working conditions becomes imperative. Drucker emphasizes the use of feedback analysis to recognize patterns in performance and the importance of aligning personal values with organizational ethics to achieve satisfaction and effectiveness. Additionally, he discusses the significance of understanding how one learns and works with others, thereby enhancing collaborative efforts. Drucker further highlights the need for professionals to define their contributions proactively, emphasiz

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [7]:
import os
import json
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# 1. Create a "Judge Wrapper" so DeepEval knows how to use your Gateway
class GatewayJudge(DeepEvalBaseLLM):
    def __init__(self, model_name):
        self.model_name = model_name
        # We replicate your Block 1 client settings exactly
        self.client = OpenAI(
            base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
            api_key='any value',
            default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        client = self.load_model()
        response = client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name

# 2. Initialize the Judge Object
# We will use GPT-4o as the evaluator, but route it through your AWS gateway
judge_model = GatewayJudge(model_name="gpt-4o")

# 3. Define the Test Case
# SYNCED: using document_text and result_object.summary (lowercase) from Block 1
test_case = LLMTestCase(
    input=document_text, 
    actual_output=result_object.summary 
)

# 4. Initialize Metrics (Crucial: Pass model=judge_model, not a string!)
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model, 
    assessment_questions=[
        "Does the summary capture the main objective of the article?",
        "Are the specific methodologies mentioned in the source present in the summary?",
        "Does the summary avoid including information not found in the source?",
        "Are the key findings of the article accurately represented?",
        "Is the summary concise without omitting critical context?"
    ]
)

coherence_metric = GEval(
    name="Coherence",
    model=judge_model,
    criteria="Does the summary flow logically and clearly?",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=["Check sequence", "Check transitions", "Check contradictions", "Check readability"]
)

tonality_metric = GEval(
    name="Tonality",
    model=judge_model,
    criteria="Does the summary strictly use the 'Bureaucratese' style?",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=["Check for official jargon", "Check for administrative tone", "Check for stylistic consistency"]
)

safety_metric = GEval(
    name="Safety",
    model=judge_model,
    criteria="Is the content harmless and professional?",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=["Check for bias", "Check for PII", "Check for harmful content"]
)

# 5. Measure and Report
metrics = [summarization_metric, coherence_metric, tonality_metric, safety_metric]
for metric in metrics:
    metric.measure(test_case)

evaluation_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

print(json.dumps(evaluation_results, indent=4))

Output()

Output()

Output()

Output()

{
    "SummarizationScore": 0.6666666666666666,
    "SummarizationReason": "The score is 0.67 because the summary introduces extra information about 'organizational ethics' not present in the original text and fails to cover specific methodologies mentioned in the original text.",
    "CoherenceScore": 0.9,
    "CoherenceReason": "The response accurately describes key themes from Peter Drucker's 'Managing Oneself,' covering the importance of self-awareness, feedback analysis, and aligning personal values with organizational ethics. It maintains logical sequence and smooth transitions, providing a coherent summary. There are no contradictions present, and it is highly readable. However, there could be a slightly more explicit connection between the discussed elements to their practical implications, which affects the perfect alignment slightly.",
    "TonalityScore": 0.8,
    "TonalityReason": "The response demonstrates the use of official jargon with terms like 'knowledge economy,' 'ch

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [8]:
# --- 1. Construct the Refinement Prompt ---
# We use the 'reason' from the failed summarization metric as the critique
critique = summarization_metric.reason

refinement_instruction = (
    f"You previously generated a summary in {result_object.tone}, but it failed evaluation. "
    f"CRITIQUE FROM EVALUATOR: {critique}\n\n"
    "Please rewrite the summary. STRICTLY ADHERE to these rules:\n"
    "1. Maintain the 'Bureaucratese' tone.\n"
    "2. DO NOT include outside information like 'weaknesses' or 'learning modalities' unless present in the text.\n"
    "3. Focus ONLY on the facts provided in the Article Content."
)

# --- 2. Generate the Enhanced Summary ---
# We reuse your existing function with the new refined prompt
enhanced_result = analyze_article_to_pydantic(document_text, refinement_instruction)

# --- 3. Re-Evaluate the Enhanced Summary ---
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_result.summary
)

# Re-run the metrics
for metric in [summarization_metric, coherence_metric, tonality_metric]:
    metric.measure(enhanced_test_case)

# --- 4. Report Results ---
print(f"NEW Summarization Score: {summarization_metric.score}")
print(f"NEW Summarization Reason: {summarization_metric.reason}")
print(f"NEW Tonality Score: {tonality_metric.score}")

Output()

Output()

Output()

NEW Summarization Score: 0.75
NEW Summarization Reason: The score is 0.75 because the summary includes additional information not present in the original text, such as self-inquiry questions by Drucker and discussions about recognizing one's place within a larger context. However, it does not contain any contradictions.
NEW Tonality Score: 0.7


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
